# RiskModels for Omega Point

### A live API session for a partner contact

**Goal:** evaluate Orth Risk as a **hosted model** you could offer alongside Barra and other models you already distribute — not as a competing UI or as “our proprietary risk model.”

RiskModels delivers **ERM3 Orth Risk** over HTTPS only in this notebook:

| Layer | What it isolates | What you can trade / study |
|---|---|---|
| Market | broad equity exposure | size a market ETF hedge |
| Sector | incremental sector exposure | industry beta vs market beta |
| Subsector | incremental business-model exposure | refine hedge + peers |
| Residual / idiosyncratic | what remains after the hierarchy | stock-specific or manager-selection sleeve |

**This session is API-only.** Pre-rendered snapshot panels and server PDFs come from the RiskModels API. Long-history zarr access is out of scope here (can be permissioned later).

**Docs:** [API guide](https://riskmodels.app/docs/api) · [Interactive reference](https://riskmodels.app/api-reference) · [Get an API key](https://riskmodels.app/get-key)

**Also runnable as a CLI:** `python/stocks_and_funds_quickstart.py` (see repo `README.md`).


## 0 · Install once

In Google Colab, run the next cell once. Locally, install the same packages in your environment.


In [ ]:
import sys
import subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "riskmodels-py", "pandas", "matplotlib", "requests", "python-dotenv",
    ])
    print("Installed RiskModels and notebook dependencies.")
else:
    print("Local notebook: pip install -U riskmodels-py pandas matplotlib requests python-dotenv")


## 1 · Connect securely

`quickstart_connect()` looks for `RISKMODELS_API_KEY` in the environment, a local `.env` / `.env.local`, or Colab Secrets. If missing, it prompts securely. The key is never printed.


In [ ]:
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from IPython.display import Image, display

from riskmodels import RiskModelsClient
from riskmodels.notebook import quickstart_connect

session, BASE_URL, API_KEY = quickstart_connect()
os.environ.setdefault("RISKMODELS_API_KEY", API_KEY)
os.environ.setdefault("RISKMODELS_BASE_URL", BASE_URL)
client = RiskModelsClient.from_env()

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

COLORS = {
    "market": "#4F46E5",
    "sector": "#16A34A",
    "subsector": "#0EA5E9",
    "idiosyncratic": "#6B7280",
    "gross": "#111827",
}

def api_get(path: str, **params: Any) -> tuple[dict, dict]:
    response = session.get(f"{BASE_URL}{path}", params=params, timeout=90)
    if not response.ok:
        detail = response.text[:500]
        raise RuntimeError(f"GET {path} failed ({response.status_code}): {detail}")
    headers = {
        "data_as_of": response.headers.get("X-Data-As-Of"),
        "filing_date": response.headers.get("X-Data-Filing-Date"),
        "model_version": response.headers.get("X-Risk-Model-Version"),
        "cost_usd": response.headers.get("X-API-Cost-USD"),
    }
    return response.json(), headers

def first_value(series: pd.Series, *names: str, default=None):
    for name in names:
        if name in series.index and pd.notna(series[name]):
            return series[name]
    return default

def pick_col(frame: pd.DataFrame, *names: str) -> str | None:
    return next((name for name in names if name in frame.columns), None)

print("Connected to", BASE_URL)
print("Artifacts will save under", OUTPUT_DIR.resolve())


## 2 · Choose subjects

Change these and rerun. Keep the demo book small — one batch call.


In [ ]:
STOCK_TICKER = "NVDA"   # Try AAPL, JPM, TSLA
STOCK_YEARS = 3
FUND_QUERY = "AGTHX"    # Ticker or part of a mutual-fund name
NOTIONAL_USD = 10_000_000  # Dollar hedge ticket size

# Tiny "bring your book" weights — what an Omega workflow would POST
BOOK = {"NVDA": 0.25, "AAPL": 0.25, "MSFT": 0.25, "JPM": 0.25}

# Pre-rendered / server panel tiles (API). `_full` is the composed Deep Dive page.
PANEL_SLUGS = (
    "l3_explained_risk_hbar",
    "hedge_notionals_hbar",
    "hedge_depth_retained",
)

print({
    "stock": STOCK_TICKER,
    "stock_years": STOCK_YEARS,
    "fund_query": FUND_QUERY,
    "notional_usd": NOTIONAL_USD,
    "book": BOOK,
})


# Part 0 · Product surfaces (what you would host)

These calls return **bytes from the API** — pre-rendered or server-rendered panels/PDFs. No local risk engine. Outside the hot ticker cohort, some institutional units may return 501; the product panels below are the general offer.


In [ ]:
panel_paths = []
for slug in PANEL_SLUGS:
    try:
        payload, lineage = client.snapshot_panel("stock", STOCK_TICKER, slug, format="png")
        if not isinstance(payload, (bytes, bytearray)):
            print(f"{slug}: unexpected payload type {type(payload)}; skip display")
            continue
        out = OUTPUT_DIR / f"{STOCK_TICKER}_{slug}.png"
        out.write_bytes(payload)
        panel_paths.append(out)
        print(f"OK  {slug}  ({len(payload):,} bytes)  as_of={getattr(lineage, 'data_as_of', None)}  → {out.name}")
        display(Image(data=bytes(payload)))
    except Exception as exc:
        print(f"SKIP {slug}: {exc}")

try:
    pdf_bytes, pdf_lineage = client.get_metrics_snapshot_pdf(STOCK_TICKER)
    pdf_path = OUTPUT_DIR / f"{STOCK_TICKER}_metrics_snapshot.pdf"
    pdf_path.write_bytes(pdf_bytes)
    print(f"PDF {pdf_path.name} ({len(pdf_bytes):,} bytes) — forwardable tearsheet")
except Exception as exc:
    print(f"SKIP metrics snapshot PDF: {exc}")


# Part I · Stock (numbers behind the tiles)

## 3 · Current risk: four variance shares, three ETF legs, dollar ticket

**Explained Risk (ER)** is a variance share. At L3, market + sector + subsector + residual ≈ 100%.

**Hedge Ratio (HR)** is ETF dollars per $1 of stock. Negatives are allowed under orthogonalization.

Scale HR by `NOTIONAL_USD` for a trade ticket a PM recognizes.


In [ ]:
metrics_df = client.get_metrics(STOCK_TICKER, as_dataframe=True)
if metrics_df.empty:
    raise ValueError(f"No metrics returned for {STOCK_TICKER}.")

row = metrics_df.iloc[-1]

er = pd.Series({
    "Market": first_value(row, "l3_market_er", "l3_mkt_er"),
    "Sector": first_value(row, "l3_sector_er", "l3_sec_er"),
    "Subsector": first_value(row, "l3_subsector_er", "l3_sub_er"),
    "Residual": first_value(row, "l3_residual_er", "l3_res_er"),
}, dtype="float64")

hedge = pd.DataFrame([
    {
        "Layer": "Market",
        "ETF": first_value(row, "market_factor_etf", "market_etf", default="SPY"),
        "HR ($ ETF / $1 stock)": first_value(row, "l3_market_hr", "l3_mkt_hr"),
    },
    {
        "Layer": "Sector",
        "ETF": first_value(row, "sector_etf", default="sector ETF"),
        "HR ($ ETF / $1 stock)": first_value(row, "l3_sector_hr", "l3_sec_hr"),
    },
    {
        "Layer": "Subsector",
        "ETF": first_value(row, "subsector_etf", default="subsector ETF"),
        "HR ($ ETF / $1 stock)": first_value(row, "l3_subsector_hr", "l3_sub_hr"),
    },
])
hedge["Notional hedge ($)"] = hedge["HR ($ ETF / $1 stock)"] * NOTIONAL_USD

snapshot = pd.DataFrame({
    "Value": {
        "Ticker": STOCK_TICKER,
        "As of": first_value(row, "date", "teo", default="latest"),
        "Price": first_value(row, "price_close"),
        "23d annualized volatility": first_value(row, "vol_23d"),
        "L3 ER total": er.sum(min_count=1),
        "Ticket notional": NOTIONAL_USD,
    }
})

display(snapshot)
hedge_display = hedge.copy()
hedge_display["HR ($ ETF / $1 stock)"] = hedge_display["HR ($ ETF / $1 stock)"].map(
    lambda x: f"{x:.3f}" if pd.notna(x) else "—"
)
hedge_display["Notional hedge ($)"] = hedge_display["Notional hedge ($)"].map(
    lambda x: f"${x:,.0f}" if pd.notna(x) else "—"
)
display(hedge_display)

fig, ax = plt.subplots(figsize=(9, 3.8))
colors = [COLORS["market"], COLORS["sector"], COLORS["subsector"], COLORS["idiosyncratic"]]
er.dropna().plot(kind="barh", ax=ax, color=colors[: er.notna().sum()])
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlim(left=0)
ax.set_xlabel("Share of variance")
ax.set_ylabel("")
ax.set_title(f"{STOCK_TICKER} — current L3 Orth Risk decomposition")
ax.grid(axis="x", alpha=0.2)
for container in ax.containers:
    ax.bar_label(container, labels=[f"{v:.1%}" for v in er.dropna()], padding=4)
plt.tight_layout()
plt.show()


## 4 · Historical returns: same hierarchy through time

Arithmetic cumulative attribution (`cumsum`) — additive layers, not compounded investor return. Still API-only (`get_ticker_returns`).


In [ ]:
stock_history = client.get_ticker_returns(STOCK_TICKER, years=STOCK_YEARS).copy()
if stock_history.empty:
    raise ValueError(f"No return history returned for {STOCK_TICKER}.")

stock_history["date"] = pd.to_datetime(stock_history["date"])
stock_history = stock_history.sort_values("date").reset_index(drop=True)

gross_col = pick_col(stock_history, "returns_gross", "gross_return")
l1_fr = pick_col(stock_history, "l1_factor_return", "l1_fr")
l2_fr = pick_col(stock_history, "l2_factor_return", "l2_fr")
l3_fr = pick_col(stock_history, "l3_factor_return", "l3_fr")
l1_cfr = pick_col(stock_history, "l1_combined_factor_return", "l1_cfr")
l2_cfr = pick_col(stock_history, "l2_combined_factor_return", "l2_cfr")
l3_cfr = pick_col(stock_history, "l3_combined_factor_return", "l3_cfr")
l3_rr = pick_col(stock_history, "l3_residual_return", "l3_rr")

if gross_col is None or l3_cfr is None:
    raise KeyError("Expected gross and L3 factor-return fields are not present. Inspect stock_history.columns.")

contrib = pd.DataFrame(index=stock_history.index)
contrib["Market"] = stock_history[l1_fr] if l1_fr else stock_history[l1_cfr]
contrib["Sector"] = stock_history[l2_fr] if l2_fr else stock_history[l2_cfr] - stock_history[l1_cfr]
contrib["Subsector"] = stock_history[l3_fr] if l3_fr else stock_history[l3_cfr] - stock_history[l2_cfr]
contrib["Idiosyncratic"] = stock_history[l3_rr] if l3_rr else stock_history[gross_col] - stock_history[l3_cfr]
contrib["Gross"] = stock_history[gross_col]

cumulative = contrib.fillna(0).cumsum()
fig, ax = plt.subplots(figsize=(11, 5))
for label, key in [
    ("Market", "market"), ("Sector", "sector"),
    ("Subsector", "subsector"), ("Idiosyncratic", "idiosyncratic"),
]:
    ax.plot(stock_history["date"], cumulative[label], label=label, color=COLORS[key], linewidth=1.6)
ax.plot(stock_history["date"], cumulative["Gross"], label="Gross", color=COLORS["gross"], linewidth=2.2, linestyle="--")
ax.axhline(0, color="#9CA3AF", linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title(f"{STOCK_TICKER} — arithmetic cumulative return attribution ({STOCK_YEARS}y)")
ax.set_ylabel("Cumulative contribution")
ax.legend(ncol=5, frameon=False, loc="upper left")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

check = (contrib[["Market", "Sector", "Subsector", "Idiosyncratic"]].sum(axis=1) - contrib["Gross"]).abs()
print("Mean absolute daily identity gap:", f"{check.mean():.6%}")


### Stock readout

1. **Where is the risk?** ER tiles / chart separate systematic structure from residual.
2. **What would I trade?** HR × notional → ETF dollar ticket.
3. **What drove the path?** Historical split applies the same hierarchy to realized returns.


# Part II · Fund capabilities

## 5 · Resolve a fund

Fund search is free and returns a stable `bw_fund_id` for analytics endpoints.


In [ ]:
search_body, search_headers = api_get("/funds/search", q=FUND_QUERY, limit=10)
fund_hits = search_body.get("results") or []
if not fund_hits:
    raise ValueError(f"No fund matched {FUND_QUERY!r}. Try a ticker or part of the fund name.")

hits = pd.DataFrame(fund_hits)
show_cols = [
    c for c in ["ticker", "fund_name", "equity_style_9box", "morningstar_category",
                "net_expense_ratio", "latest_report_date", "bw_fund_id"]
    if c in hits.columns
]
display(hits[show_cols].head(10))

exact = hits[hits.get("ticker", pd.Series(index=hits.index, dtype=str)).astype(str).str.upper() == FUND_QUERY.upper()]
selected = (exact.iloc[0] if not exact.empty else hits.iloc[0]).to_dict()
FUND_ID = selected["bw_fund_id"]
FUND_LABEL = selected.get("ticker") or selected.get("fund_name") or FUND_ID

print("Selected:", FUND_LABEL, "|", selected.get("fund_name"), "|", FUND_ID)
print("Search lineage / cost headers:", {k: v for k, v in search_headers.items() if v})


## 6 · Composed fund snapshot

One call bundles registry metadata, bitemporal dates, holdings-derived returns, diagnostics, history, peers, and hedge basket — the fund analogue of Orth Risk.


In [ ]:
fund_snapshot, fund_headers = api_get(f"/funds/snapshot/{FUND_ID}")

fund_metrics = fund_snapshot.get("metrics") or {}
fund_returns = fund_metrics.get("returns") or {}
diagnostics = fund_metrics.get("diagnostics") or {}
metadata = fund_snapshot.get("_metadata") or fund_metrics.get("_metadata") or {}

identity = pd.DataFrame({
    "Value": {
        "Fund": fund_snapshot.get("fund_name") or selected.get("fund_name"),
        "Ticker": fund_snapshot.get("ticker") or selected.get("ticker"),
        "Style cell": fund_snapshot.get("equity_style_9box") or selected.get("equity_style_9box"),
        "Holdings report date": fund_snapshot.get("report_date") or fund_headers.get("data_as_of"),
        "Filing date": fund_snapshot.get("filing_date") or fund_headers.get("filing_date"),
        "Model version": metadata.get("model_version") or fund_headers.get("model_version"),
        "API cost for this call": fund_headers.get("cost_usd"),
    }
})
display(identity)

latest_components = pd.Series({
    "Market": fund_returns.get("market"),
    "Sector": fund_returns.get("sector"),
    "Subsector": fund_returns.get("subsector"),
    "Idiosyncratic": fund_returns.get("idiosyncratic"),
}, dtype="float64")
gross = fund_returns.get("gross")

fig, ax = plt.subplots(figsize=(9, 3.8))
bars = latest_components.dropna()
ax.bar(bars.index, bars.values, color=[
    COLORS["market"], COLORS["sector"], COLORS["subsector"], COLORS["idiosyncratic"]
][: len(bars)])
ax.axhline(0, color="#9CA3AF", linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title(f"{FUND_LABEL} — latest holdings-derived return decomposition")
ax.set_ylabel("Return contribution")
ax.grid(axis="y", alpha=0.2)
for container in ax.containers:
    ax.bar_label(container, labels=[f"{v:.2%}" for v in bars.values], padding=3)
plt.tight_layout()
plt.show()

if gross is not None:
    explained_sum = latest_components.sum(min_count=1)
    print("Gross return:", f"{gross:.2%}", "| Layer sum:", f"{explained_sum:.2%}",
          "| Identity gap:", f"{(gross - explained_sum):.4%}")


## 7 · Is the stock-selection sleeve persistent?

Idiosyncratic series = holdings-weighted stock-specific sleeve after market / sector / subsector.


In [ ]:
history_rows = ((fund_snapshot.get("portfolio_history") or {}).get("rows") or [])
fund_history = pd.DataFrame(history_rows)

if fund_history.empty:
    print("No portfolio history in this fund snapshot. Try another fund with a longer holdings panel.")
else:
    fund_history["teo"] = pd.to_datetime(fund_history["teo"])
    fund_history = fund_history.sort_values("teo").reset_index(drop=True)

    rename = {
        "portfolio_market_return": "Market",
        "portfolio_sector_return": "Sector",
        "portfolio_subsector_return": "Subsector",
        "portfolio_idiosyncratic_return": "Idiosyncratic",
        "portfolio_gross_return": "Gross",
    }
    available = {k: v for k, v in rename.items() if k in fund_history.columns}
    fund_attr = fund_history[list(available)].rename(columns=available).fillna(0).cumsum()

    fig, ax = plt.subplots(figsize=(11, 5))
    for label, key in [
        ("Market", "market"), ("Sector", "sector"),
        ("Subsector", "subsector"), ("Idiosyncratic", "idiosyncratic"),
    ]:
        if label in fund_attr:
            ax.plot(fund_history["teo"], fund_attr[label], label=label, color=COLORS[key], linewidth=1.8)
    if "Gross" in fund_attr:
        ax.plot(fund_history["teo"], fund_attr["Gross"], label="Gross", color=COLORS["gross"], linewidth=2.2, linestyle="--")
    ax.axhline(0, color="#9CA3AF", linewidth=0.8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f"{FUND_LABEL} — arithmetic cumulative fund attribution")
    ax.set_ylabel("Cumulative contribution")
    ax.legend(ncol=5, frameon=False, loc="upper left")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

    if "portfolio_idiosyncratic_return" in fund_history:
        resid = fund_history["portfolio_idiosyncratic_return"].dropna()
        persistence = pd.DataFrame({
            "Value": {
                "Observed months": len(resid),
                "Positive idiosyncratic months": (resid > 0).mean() if len(resid) else np.nan,
                "Arithmetic cumulative idiosyncratic contribution": resid.sum(),
            }
        })
        display(persistence)


## 8 · Context: coverage, concentration, holdings, peer rank

Guardrails around any “skill” read.


In [ ]:
diagnostic_table = pd.DataFrame({
    "Value": {
        "ERM3 universe coverage": diagnostics.get("weight_sum"),
        "Active holdings": diagnostics.get("n_holdings_active"),
        "Effective N (HHI)": diagnostics.get("effective_n"),
        "Top-10 weight": diagnostics.get("top10_weight_sum"),
    }
})
display(diagnostic_table)

holdings_block = fund_snapshot.get("holdings") or {}
top_holdings = pd.DataFrame(holdings_block.get("top") or [])
if not top_holdings.empty:
    holding_cols = [c for c in ["bw_sym_id", "weight", "adj_mv"] if c in top_holdings.columns]
    holdings_display = top_holdings[holding_cols].head(10).copy()
    if "weight" in holdings_display:
        holdings_display["weight"] = holdings_display["weight"].map(
            lambda x: f"{x:.2%}" if pd.notna(x) else "—"
        )
    if "adj_mv" in holdings_display:
        holdings_display["adj_mv"] = holdings_display["adj_mv"].map(
            lambda x: f"${x:,.0f}" if pd.notna(x) else "—"
        )
    display(holdings_display)
else:
    print("No holdings block in this snapshot.")

ranks = pd.DataFrame(((fund_snapshot.get("cohort_context") or {}).get("ranks") or []))
if not ranks.empty:
    if {"rank", "cohort_size"}.issubset(ranks.columns):
        ranks["percentile"] = np.where(
            ranks["cohort_size"].fillna(0) > 1,
            100 * (1 - (ranks["rank"] - 1) / ranks["cohort_size"]),
            np.nan,
        )
    rank_cols = [c for c in ["metric", "period_window", "rank", "cohort_size", "percentile", "value"] if c in ranks.columns]
    display(ranks[rank_cols].sort_values([c for c in ["metric", "period_window"] if c in rank_cols]))
else:
    print("No peer ranks are available for this fund/style-date combination.")


## 9 · Fund hedge basket (+ dollar ticket)

Same HR convention as stocks: ETF notional per $1 of fund exposure, scaled by `NOTIONAL_USD`.


In [ ]:
hedge_block = fund_snapshot.get("hedge") or {}
hedge_rows = []
for level in ["L1", "L2", "L3"]:
    for leg in hedge_block.get(level) or []:
        hedge_rows.append({"Level": level, "ETF": leg.get("etf"), "HR ($ ETF / $1 fund)": leg.get("hr")})

fund_hedge = pd.DataFrame(hedge_rows)
if fund_hedge.empty:
    print("No fund hedge basket is available for this snapshot.")
else:
    fund_hedge = fund_hedge.copy()
    fund_hedge["Notional hedge ($)"] = fund_hedge["HR ($ ETF / $1 fund)"] * NOTIONAL_USD
    fund_hedge_display = fund_hedge.copy()
    fund_hedge_display["HR ($ ETF / $1 fund)"] = fund_hedge_display["HR ($ ETF / $1 fund)"].map(
        lambda x: f"{x:.4f}" if pd.notna(x) else "—"
    )
    fund_hedge_display["Notional hedge ($)"] = fund_hedge_display["Notional hedge ($)"].map(
        lambda x: f"${x:,.0f}" if pd.notna(x) else "—"
    )
    display(fund_hedge_display)

    plotted = fund_hedge.dropna(subset=["HR ($ ETF / $1 fund)"])
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.32 * len(plotted))))
    labels = plotted["Level"] + " · " + plotted["ETF"].astype(str)
    colors = plotted["Level"].map({"L1": COLORS["market"], "L2": COLORS["sector"], "L3": COLORS["subsector"]})
    ax.barh(labels, plotted["HR ($ ETF / $1 fund)"], color=colors)
    ax.axvline(0, color="#9CA3AF", linewidth=0.8)
    ax.set_xlabel("ETF notional per $1 of fund exposure")
    ax.set_title(f"{FUND_LABEL} — latest fund hedge basket")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()


# Part III · Bring your book

Omega Point already pipes client portfolios through hosted models. This cell POSTs a tiny weight vector to RiskModels — the same pattern you would use to host Orth Risk next to Barra.


In [ ]:
pa = client.analyze_portfolio(BOOK, metrics=["full_metrics", "hedge_ratios"], years=1)
print("Book:", BOOK)

# Portfolio-level ER / HR aggregates when present
er_port = getattr(pa, "portfolio_l3_er_weighted_mean", None)
hr_port = getattr(pa, "portfolio_hedge_ratios", None)
if er_port is not None:
    display(pd.DataFrame({"L3 ER (weight-mean)": er_port}).T if not isinstance(er_port, pd.DataFrame) else er_port)
if hr_port is not None:
    display(hr_port if isinstance(hr_port, pd.DataFrame) else pd.DataFrame({"portfolio_hedge_ratios": hr_port}))

per = getattr(pa, "per_ticker", None)
if per is not None:
    if isinstance(per, pd.DataFrame):
        cols = [c for c in per.columns if any(k in c for k in ("er", "hr", "ticker", "residual"))]
        display(per[cols].head(20) if cols else per.head(20))
    else:
        print(per)

try:
    book_pdf, book_lineage = client.post_portfolio_risk_snapshot_pdf(BOOK, title="Omega demo book")
    book_pdf_path = OUTPUT_DIR / "demo_book_risk_snapshot.pdf"
    book_pdf_path.write_bytes(book_pdf)
    print(f"Book PDF → {book_pdf_path} ({len(book_pdf):,} bytes)")
except TypeError:
    # Older SDK may not accept title=
    try:
        book_pdf, book_lineage = client.post_portfolio_risk_snapshot_pdf(BOOK)
        book_pdf_path = OUTPUT_DIR / "demo_book_risk_snapshot.pdf"
        book_pdf_path.write_bytes(book_pdf)
        print(f"Book PDF → {book_pdf_path} ({len(book_pdf):,} bytes)")
    except Exception as exc:
        print(f"SKIP book PDF: {exc}")
except Exception as exc:
    print(f"SKIP book PDF: {exc}")


# Takeaways for a model host

1. **Orth Risk is a model to host**, not a replacement for Omega Point’s platform or for Barra. Same multi-model shelf pattern you already run.
2. **What clients get that factor dumps often lack:** hierarchical ER shares **and** signed ETF hedge ratios (stock, fund, and small book) over a stable API — plus pre-rendered panels / PDFs you can surface in existing workflows.
3. **API-only today.** Snapshot panels and server PDFs are the product visual surface. Longer zarr history can be permissioned later; it is not required for a client-facing integration.
4. **Interpretation discipline:** residual / idiosyncratic is stock selection *after* the hierarchy — not proof of skill by itself.

**Question for the room:** Would you host Orth Risk alongside Barra for clients who want this equity + holdings-derived fund residual view?
